In [3]:
from IPython.display import display, clear_output
import os
import sourcedefender
import numpy as np
import torch
import sys
import argparse
import time
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from tqdm import tqdm
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, random_split, dataset, Subset
import pandas as pd
import argparse
from mpl_toolkits.mplot3d import Axes3D
import gc

sys.path.append("../../")
sys.path.append("../")
sys.path.append("./")

from lib.batchJacobian import batchJacobian_PDE
from lib.util import MHPI, UnitGaussianNormalizer
from lib.utilities3 import prepare_dataset, upscale_tensor, calculate_errors, check_if_from_ddp, adjust_state_dict, plot_log_loss, NonLocalMeansSmoothing, calculate_relative_errors
from lib.utiltools import loss_live_plot, GaussianRandomFieldGenerator, generate_batch_parameters, AutomaticWeightedLoss
from lib.DerivativeComputer import batchJacobian_AD_Dist
from lib.hellper import *
from models.fno3d_encoder import FNO3d
from lib.helper import LargeHydrologyDataset


# Loading Dataset

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f'Device: {device}\n')

# Dataset and case configuration
DATASET_PATH = '/storage/group/cxs1024/default/mehdi/dam_brek/scenario_groups_1/'
SAVED_MODEL_PATH = '../../experiments/Hurricane_Matthew/saved_models'
# This creates a 'pointer' to the data on disk without filling up your RAM

train_size = 300
eval_size = 150
batch_size = 2

enable_ig_loss = False
tag = 'test_coarse3'
case = 'Hurricane_Matthew'
operator_type = 'FNO'


if "coarse" in tag:
    Nx = 164 * 2
    Ny = 164
    topo_path = DATASET_PATH + 'hurricane_matthew_processed_data_bed_2.pt'
    a_path = DATASET_PATH + "hurricane_matthew_processed_data_input_2.pt"
    u_path = DATASET_PATH + "hurricane_matthew_processed_data_solution_2.pt"

else:
    Nx = 313
    Ny = 158
    topo_path = DATASET_PATH + 'hurricane_matthew_processed_data_bed.pt'
    a_path = DATASET_PATH + "hurricane_matthew_processed_data_input.pt"
    u_path = DATASET_PATH + "hurricane_matthew_processed_data_solution.pt"


T_Total = 89
T_in = 1
T_out = T_Total - T_in


ig_tag = "IG_Enable" if enable_ig_loss else "IG_Disable"

parts = [
    f"saved_model_{case}",
    ig_tag,
    f"Nx_{Nx}", f"Ny_{Ny}",
    f"Tin_{T_in}", f"Tout_{T_out}",
    f"Samp_{tag}",
    f"{operator_type}_DDP_{train_size}"
]

MODE = "_".join(parts[1:])
checkpoint_path = os.path.join(SAVED_MODEL_PATH, "_".join(parts) + ".pth")

full_dataset = LargeHydrologyDataset(a_path, u_path)
# 2. Split with a fixed generator for reproducibility
n = len(full_dataset)

# Make deterministic indices
g = torch.Generator().manual_seed(42)
perm = torch.randperm(n, generator=g).tolist()
test_idx  = perm[train_size + eval_size:]
test_dataset = Subset(full_dataset, test_idx)

test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
plot_log_loss(checkpoint["loss_df"], mode=MODE, plot_ig_loss=False)

Device: cuda:0



KeyError: 'loss_df'

# Define FNO model

In [ ]:
checkpoint['config']

In [ ]:
if "coarse" in tag:
    from models.fno3d_encoder import FNO3d
else:
    from models.fno3d import FNO3d

is_DDP = check_if_from_ddp(checkpoint)
model = FNO3d(
    T_in=T_in, T_out=T_out,
    modes_x=checkpoint['config']['mode1'], 
    modes_y=checkpoint['config']['mode2'], 
    modes_t=checkpoint['config']['mode3'],
    width=checkpoint['config']['width_FNO'],
    
    encoder_kernel_size_x=82,
    encoder_kernel_size_y=41,
    encoder_num_layers=4
)
    
if is_DDP:   
    adjusted_state_dict = adjust_state_dict(checkpoint['model_state_dict'], model)
    model.load_state_dict(adjusted_state_dict)
else:
    model.load_state_dict(checkpoint['model_state_dict'])

model = model.to(device)


# Forward run the trained model for evaluation

In [ ]:
import time
import torch
from tqdm import tqdm
from lib.helper import coarsen_spatial_tensor

model.eval()

# ---- user control ----
MAX_SAMPLES = 100   # <- set this to how many samples you want

# ---- load topo once ----
topo = torch.load(topo_path, map_location="cpu").to(device)

# ---- CPU buffers ----
forcing_all = torch.empty(0, device="cpu")
u0_all      = torch.empty(0, device="cpu")
u_true_all  = torch.empty(0, device="cpu")
u_pred_all  = torch.empty(0, device="cpu")

t_start = time.time()
num_collected = 0

with torch.no_grad():
    for _, batch_data in tqdm(enumerate(test_loader, 1), total=len(test_loader),
                              desc="Processing batches"):

        batch_data = [item.to(device, non_blocking=True) for item in batch_data]
        batch_forcing = batch_data[0]
        batch_sol     = batch_data[1]

        batch_u0    = batch_sol[..., :T_in]
        batch_u_out = batch_sol[..., T_in:]

        bs = batch_u0.shape[0]

        # ---- trim batch if it would exceed MAX_SAMPLES ----
        remaining = MAX_SAMPLES - num_collected
        if remaining <= 0:
            break
        if bs > remaining:
            batch_forcing = batch_forcing[:remaining]
            batch_u0      = batch_u0[:remaining]
            batch_u_out   = batch_u_out[:remaining]
            bs = remaining

        batch_topo = topo.expand(bs, -1, -1)

        U_pred_batch = model(batch_forcing, batch_u0, batch_topo)

        # ---- move to CPU ----
        forcing_all = torch.cat((forcing_all, batch_forcing.cpu()), dim=0)
        u0_all      = torch.cat((u0_all, batch_u0.cpu()), dim=0)
        u_true_all  = torch.cat((u_true_all, batch_u_out.cpu()), dim=0)
        u_pred_all  = torch.cat((u_pred_all, U_pred_batch.cpu()), dim=0)

        num_collected += bs

        del batch_forcing, batch_sol, batch_u0, batch_u_out, batch_topo, U_pred_batch
        
u_pred_all[u_pred_all<0.025] = 0.0
u_true_all[u_true_all<0.025] = 0.0

t_end = time.time()
print(
    f"Processed {num_collected} samples in {t_end - t_start:.2f}s "
    f"({(t_end - t_start)/max(num_collected,1):.3f} s/sample)"
)

if "coarse" in tag:
    u_true_all = coarsen_spatial_tensor(u_true_all, N=checkpoint['config']['magnification_factor'],  mode='bilinear')


In [ ]:
checkpoint['config']

In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from datetime import datetime, timedelta
import numpy as np

# =========================
# 0) Pick 2 samples
# =========================

assert u_true_all.shape == u_pred_all.shape, "u_true_all and u_pred_all must have same shape"

N = u_true_all.shape[0]
idx = torch.randperm(N)[:2].tolist()  # choose 2 random samples

u_true_2 = u_true_all[idx].detach().cpu().numpy()  # [2, nx, ny, nt]
u_pred_2 = u_pred_all[idx].detach().cpu().numpy()  # [2, nx, ny, nt]
u_err_2  = np.abs(u_true_2 - u_pred_2)             # [2, nx, ny, nt]

num_frames = u_true_2.shape[-1]

# =========================
# 1) Time configuration
# =========================
start_date = datetime(2016, 10, 5, 12, 0, 0)
time_step_seconds = 3600
time_dates = [start_date + timedelta(seconds=i * time_step_seconds) for i in range(num_frames)]

# =========================
# 2) Shared color scales
# =========================
vmin_global = 0.0
vmax_global = max(u_true_2.max(), u_pred_2.max()) * 0.7

err_vmin = 0.0
# err_vmax = u_err_2.max() * 0.7
err_vmax = 0.2


X_range = 248417.934 - 201527.934
Y_range = 3922540.192 - 3898940.192

extent = [0, X_range, 0, Y_range]

# =========================
# 3) Figure: 2 rows x 3 cols (Pred | True | Abs Error)
# =========================
fig, axes = plt.subplots(2, 3, figsize=(15, 7), sharex=True, sharey=True)
# axes[r, c]: c=0 pred, c=1 true, c=2 error

ims = [[None, None, None], [None, None, None]]

for r in range(2):
    # --- Pred
    axp = axes[r, 0]
    z0p = u_pred_2[r, :, :, 0]
    imp = axp.imshow(
        z0p.T, extent=extent, origin="lower",
        interpolation='none',
        cmap="terrain", vmin=vmin_global, vmax=vmax_global
    )
    axp.set_title(f"Pred (sample {idx[r]})", fontsize=12)
    axp.grid(True, linestyle="--", alpha=0.3)
    ims[r][0] = imp

    # --- True
    axt = axes[r, 1]
    z0t = u_true_2[r, :, :, 0]
    imt = axt.imshow(
        z0t.T, extent=extent, origin="lower",
        interpolation='none',
        cmap="terrain", vmin=vmin_global, vmax=vmax_global
    )
    
    axt.set_title(f"True (sample {idx[r]})", fontsize=12)
    axt.grid(True, linestyle="--", alpha=0.3)
    ims[r][1] = imt

    # --- Abs Error
    axe = axes[r, 2]
    z0e = u_err_2[r, :, :, 0]
    ime = axe.imshow(
        z0e.T, extent=extent, origin="lower",
        interpolation='none',
        cmap="magma", vmin=err_vmin, vmax=err_vmax
    )
    axe.set_title(f"|True - Pred| (sample {idx[r]})", fontsize=12)
    axe.grid(True, linestyle="--", alpha=0.3)
    ims[r][2] = ime

# Row labels (optional)
axes[0, 0].set_ylabel("Sample 1")
axes[1, 0].set_ylabel("Sample 2")

# Two colorbars: one for depth, one for error
cbar_ax1 = fig.add_axes([0.10, 0.09, 0.55, 0.02])
fig.colorbar(ims[0][0], cax=cbar_ax1, label="Water Depth (m)", orientation="horizontal")

cbar_ax2 = fig.add_axes([0.72, 0.09, 0.20, 0.02])
fig.colorbar(ims[0][2], cax=cbar_ax2, label="Abs Error (m)", orientation="horizontal")

plt.subplots_adjust(bottom=0.15, hspace=0.25, wspace=0.12)

# =========================
# 4) Update function
# =========================
def update(frame_idx):
    if frame_idx < len(time_dates):
        current_dt = time_dates[frame_idx]
        fig.suptitle(
            f"Hurricane Matthew\nTime: {current_dt.strftime('%Y-%m-%d %H:%M')}",
            fontsize=16, fontweight="bold"
        )
    else:
        fig.suptitle(f"Hurricane Matthew\nFrame {frame_idx}", fontsize=16, fontweight="bold")

    for r in range(2):
        ims[r][0].set_data(u_pred_2[r, :, :, frame_idx].T)
        ims[r][1].set_data(u_true_2[r, :, :, frame_idx].T)
        ims[r][2].set_data(u_err_2[r, :, :, frame_idx].T)

    return [ims[0][0], ims[0][1], ims[0][2], ims[1][0], ims[1][1], ims[1][2]]

# =========================
# 5) Animation
# =========================

ani = animation.FuncAnimation(fig, update, frames=range(num_frames), interval=150, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import torch
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

# ─── Assumptions ─────────────────────────────────────────────────────────────
# u_true_all, u_pred_all shape: [N_samples, nx, ny, nt] or similar
# We assume first dimension is the sample dimension

assert u_true_all.shape == u_pred_all.shape
N_samples = u_true_all.shape[0]

# Convert to numpy
u_true = u_true_all.detach().cpu().numpy()    # [N, ...]
u_pred = u_pred_all.detach().cpu().numpy()

# Define depth ranges
depth_ranges = [
    (0.0,  1.0,  "Very shallow (0–1 m)"),
    (1.0,  3.0,  "Shallow (1–3 m)"),
    (3.0,  5.0,  "Moderate/Deep (3–5 m)"),
    (5.0,  np.inf, "Deep/Submerged (>5 m)"),
]

depth_ranges = [
    (0.0,  0.5,  "Shallow regions (0.0 – 0.50 m)"),
    (0.5,  u_true_all.max().item(), "Deep/Submerged regions (> 0.5 m)"),
]


fig, axes = plt.subplots(int(len(depth_ranges)/2), 2, figsize=(11, int(len(depth_ranges)/2) * 5.5))
axes = axes.flatten()

for ax_idx, (dmin, dmax, title) in enumerate(depth_ranges):
    ax = axes[ax_idx]
    
    # Lists to collect per-sample metrics for this depth range
    r2_list = []
    nse_list = []
    rmse_list = []
    valid_count = 0
    
    scatter_x = []
    scatter_y = []
    
    for i in range(N_samples):
        # Get true/pred for this sample
        true_sample = u_true[i]      # shape [nx, ny, nt] or similar
        pred_sample = u_pred[i]
        
        # Flatten spatial-temporal points
        true_flat_s = true_sample.ravel()
        pred_flat_s = pred_sample.ravel()
        
        # Mask for this depth range (based on true values)
        mask = (true_flat_s >= dmin) & (true_flat_s < dmax)
        
        if np.sum(mask) < 10:  # skip samples with too few points in range
            continue
            
        x_s = true_flat_s[mask]
        y_s = pred_flat_s[mask]
        
        # Calculate metrics for this sample in this range
        r2 = r2_score(x_s, y_s)
        rmse = np.sqrt(mean_squared_error(x_s, y_s))
        
        mean_obs = np.mean(x_s)
        nse = 1 - (np.sum((y_s - x_s)**2) / np.sum((x_s - mean_obs)**2))
        
        r2_list.append(r2)
        nse_list.append(nse)
        rmse_list.append(rmse)
        valid_count += 1
        
        # Collect points for scatter (optional subsampling per sample)
        if len(x_s) > 8000:
            sel = np.random.choice(len(x_s), 8000, replace=False)
            scatter_x.extend(x_s[sel])
            scatter_y.extend(y_s[sel])
        else:
            scatter_x.extend(x_s)
            scatter_y.extend(y_s)
    
    if valid_count == 0:
        ax.text(0.5, 0.5, "No valid samples in range", 
                ha='center', va='center', transform=ax.transAxes,
                fontsize=12, color='gray')
        ax.set_title(title)
        continue
    
    # Convert to arrays for scatter
    scatter_x = np.array(scatter_x)
    scatter_y = np.array(scatter_y)
    
    ax.scatter(scatter_x, scatter_y, s=4, alpha=0.5, color='royalblue',
               edgecolors='none', rasterized=True)
    
    # Axis limits
    if np.isfinite(dmax):
        ax.set_xlim(dmin, dmax)
        ax.set_ylim(dmin, dmax)
    else:
        data_max = max(scatter_x.max(), scatter_y.max())
        ax.set_xlim(dmin, data_max * 1.06)
        ax.set_ylim(dmin, data_max * 1.06)
    
    ax.set_aspect('equal')
    
    # Ideal line
    xmin, xmax = ax.get_xlim()
    xx = np.linspace(xmin, xmax, 201)
    ax.plot(xx, xx, 'r--', lw=1.6, label='Ideal')
    
    # ─── Show average metrics across samples ─────────────────────────────────
    metrics_text = (
        f"Per-sample averages:\n"
        f"  {'   R²':<10} = {np.mean(r2_list):.3f} ± {np.std(r2_list):.3f}\n"
        # f"  {'NSE':<5} = {np.mean(nse_list):.3f} ± {np.std(nse_list):.3f}\n"
        f"  {'   RMSE':<5} = {np.mean(rmse_list):.3f} ± {np.std(rmse_list):.3f} m"
    )
    ax.text(
        0.04, 0.96,
        metrics_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='top',
        bbox=dict(boxstyle='round,pad=0.45', 
                  facecolor='white', alpha=0.88, edgecolor='gray')
    )
    
    ax.set_title(f"{title}", fontsize=12, pad=10)
    ax.set_xlabel("True depth (m)")
    ax.set_ylabel("Predicted depth (m)")
    ax.grid(True, ls='--', alpha=0.35)
    
    if ax_idx == 0:
        ax.legend(loc='lower right', fontsize=9.5, framealpha=0.95)

# fig.suptitle("Predicted vs True Depth – by depth range", 
#              fontsize=15, fontweight='bold', y=1.02)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()